# Chương 4.1 — Sensitivity Analysis của Cụm A: Scoring Core

11 tham số: α, β, K, T_base, γ, θ, S_base, LowConfThreshold, LowConfMultiplier, ServiceBonusContent, ServiceBonusProof.

Output từ `sensitivity bench --cluster=A`. Đổi `SNAP_ID` ở cell dưới nếu chạy trên snapshot khác.

> **Lưu ý dữ liệu:** Bonus/LowConf params chỉ có hiệu lực khi snapshot đã được populate quality signals (Plan-C) và có feedback confidence < ngưỡng; trên snapshot baseline hiện tại chúng cho ảnh hưởng ≈ 0.

In [ ]:
import sys
sys.path.insert(0, '..')
from lib import setup_thesis_style, save_figure, load_oat, load_tornado, load_sobol, load_grid

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

setup_thesis_style()

SNAP_ID = "snap_20260601_151306"  # snapshot ID from `sensitivity snapshot list`
OUTPUT_DIR = f"../output/{SNAP_ID}"
CLUSTER = "A"

In [ ]:
oat = load_oat(OUTPUT_DIR, CLUSTER)
params = oat["param"].unique()
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(11, 12), sharey=False)
axes = axes.flatten()
for ax, p in zip(axes, params):
    sub = oat[oat["param"] == p]
    ax.plot(sub["value"], sub["spearman"], marker="o", label="Spearman ρ")
    ax.axhline(0.95, ls="--", color="grey", lw=0.5)
    ax.set_title(p)
    ax.set_xlabel("value")
    ax.set_ylabel("ρ (rank stability)")
    ax.set_ylim(0, 1.05)
for ax in axes[len(params):]:
    ax.axis("off")
fig.suptitle("Fig 4.1.1 — OAT rank stability per parameter (Cluster A)")
fig.tight_layout()
save_figure(fig, "4_1_1_oat_rank_stability")
plt.show()

In [ ]:
tor = load_tornado(OUTPUT_DIR, CLUSTER)
tor["total"] = tor["delta_low"] + tor["delta_high"]
tor = tor.sort_values("total")

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(tor["param"], -tor["delta_low"], color="#5b9bd5", label="Δ low (−)")
ax.barh(tor["param"], tor["delta_high"], color="#ed7d31", label="Δ high (+)")
ax.axvline(0, color="black", lw=0.5)
ax.set_title("Fig 4.1.2 — Tornado: |Δ mean composite| per parameter")
ax.set_xlabel("Δ scalar metric (mean score)")
ax.legend(loc="lower right")
fig.tight_layout()
save_figure(fig, "4_1_2_tornado")
plt.show()

In [ ]:
sob = load_sobol(OUTPUT_DIR, CLUSTER).sort_values("st", ascending=True)
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(sob["param"], sob["s1"], color="#5b9bd5", label="S1 (first-order)")
ax.barh(sob["param"], (sob["st"] - sob["s1"]).clip(lower=0), left=sob["s1"],
        color="#ed7d31", label="ST − S1 (interactions)")
ax.set_xlim(0, 1)
ax.set_xlabel("Sobol index")
ax.set_title("Fig 4.1.3 — Sobol S1 vs ST")
ax.legend()
fig.tight_layout()
save_figure(fig, "4_1_3_sobol")
plt.show()

In [ ]:
grid = load_grid(OUTPUT_DIR, CLUSTER)
param_cols = [c for c in grid.columns if c not in {"combo_id", "spearman", "kendall", "mean", "std"}]
print("Top-3 grid parameters:", param_cols)
print("\nBest by spearman:")
print(grid.nlargest(5, "spearman")[param_cols + ["spearman", "mean"]])
print("\nBest by mean composite:")
print(grid.nlargest(5, "mean")[param_cols + ["spearman", "mean"]])

if len(param_cols) == 3:
    p1, p2, p3 = param_cols
    median_p3 = grid[p3].median()
    sub = grid[np.isclose(grid[p3], median_p3, rtol=0.01)]
    pivot = sub.pivot_table(index=p1, columns=p2, values="mean")
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(pivot, cmap="viridis", annot=False, ax=ax)
    ax.set_title(f"Fig 4.1.4 — Mean composite over ({p1}, {p2}) at {p3}={median_p3:g}")
    fig.tight_layout()
    save_figure(fig, "4_1_4_grid_heatmap")
    plt.show()